#Practice with Delta Tables

## Task: Upload the people.csv file to my learning volume and list it

In [0]:
%fs ls '/Volumes/workspace/default/learning'

## Task: Create a Delta Table from the CSV file

In [0]:
# Step 1: Create a DataFrame from the CSV file
df = spark.read.csv("/Volumes/workspace/default/learning/people.csv", header=True, inferSchema=True)
display(df)

# Step 2: Create a Delta Table from the DataFrame
df.write.format("delta").saveAsTable("workspace.default.people_table")

##Task: SQL statement to select * from the Delta Table

In [0]:
%sql
select * from people_table;

##Task:
- Insert 2 new people
- Update the Alice´s salary to 50500


In [0]:
%sql
insert into people_table
values 
  (11, 'Carlos', 42, 50000),
  (12, 'Lenka', 40, 60000);

  update people_table
  set salary = 70000
  where id = 1;


In [0]:
%sql
select * from people_table order by id;


##View the table history

In [0]:
%sql
describe history people_table;

#Practice Delta Lake LakeFlow Connect techniques

## CTAS - Create Table As ![Statement](path)

In [0]:
%sql

create table customers_ctas
as
select customer_id, first_name, last_name, date_of_birth, gender
from read_files(
  '/Volumes/workspace/default/learning/customer.csv',
  format => 'csv',
  header => true,
  inferSchema => true
);

show tables;

##Upload UI technique
Steps:
- Open the catalog browser on the schema level
- Follow the steps to create a table

##COPY INTO

In [0]:
%sql
-- Create empty table
create table if not exists customers_copyinto(
  customer_id int,
  first_name string,
  last_name string,
  date_of_birth date,
  gender string);

  -- Populate from a file using copy into
  copy into customers_copyinto
  from '/Volumes/workspace/default/learning/customer.csv'
  FILEFORMAT = CSV
  FORMAT_OPTIONS ('header' = 'true', 'inferSchema' = 'true')

  -- OBS! Lo interesante de este método es que permite hacer incremental loads,
  --      simplemente anadiendo nuevos ficheros al volumen
  --      y ejecutando el mismo copy into.e
  -- En este caso, from 'path del volumen (sin indicar fichero)'

#Data Transformation Overview

##Bronze layer
Raw data ingestion

In [0]:
%sql
create table if not exists employees_bronze (
  id int,
  name string,
  country string,
  role string);

copy into employees_bronze
from '/Volumes/workspace/default/learning/employees_dataset/'
FILEFORMAT = CSV
FORMAT_OPTIONS ('header' = 'true', 'inferSchema' = 'true')

In [0]:
%sql
select * from employees_bronze;

##Silver layer
Basic transformations:
- Select id, name and country from bronze layer
- Convert role to uppercase
- Add 2 new cols: current_timestamp and current_date 

In [0]:
%sql
create table if not exists employees_silver as
select
  id,
  name,
  country,
  upper(role) as role,
  current_timestamp() as current_timestamp,
  date(current_timestamp) as current_date
from employees_bronze;

In [0]:
%sql
select * from employees_silver;

##Gold layer
Aggregate the silver table to create the gold table.
Steps:
- Create a temp_view that aggregates the total number of employees by role
-  Create a table total_roles_gold (Data Mart)

In [0]:
%sql
-- Step 1
create view temp_view as
select
  role,
  count(*) as total
from employees_silver
group by role;

select * from temp_view;
    
-- Step 2
create table if not exists total_roles_gold (
  role string,
  total int
);

insert into total_roles_gold
select * from temp_view;

select * from total_roles_gold;